# Лабораторная 1: Mushroom (OpenML)

Вариант 16. Этот ноутбук не выполнен: данные и результаты появятся только
после вашего запуска ячеек. Подготовьте окружение по README.

План: скачать исходные данные → исследовать пропуски и частоты train →
разобрать конвейер → обучить с подбором C → оценить test и сохранить модель →
зафиксировать данные через DVC и заполнить отчёт.


In [ ]:
import sys
from pathlib import Path

# Jupyter обычно запускает ядро в каталоге notebooks или в корне проекта.
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if not (project_root / "src").is_dir():
    raise RuntimeError("Откройте ноутбук из корня проекта или notebooks.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

## 1. Явная загрузка

Следующая ячейка скачивает OpenML ID 24 и сохраняет CSV. При наличии
проверенного файла повторное скачивание не требуется. Проверяется SHA-256.


In [ ]:
from src.data.download import download_dataset

raw_path = download_dataset()
print(raw_path)

## 2. Разбиение до обучения преобразований

Все 22 исходных признака категориальные; class — цель.
Естественные пропуски stalk-root заполняются модой внутри Pipeline.
Искусственные пропуски не добавляются: данные уже содержат пропуски.
Train/test разделяются 80/20 с random_state=42 и стратификацией.


In [ ]:
from src.data.prepare import load_dataset, split_dataset

frame = load_dataset()
features_train, features_test, target_train, target_test = split_dataset(frame)
print("Размер данных:", frame.shape)
print("Train/test:", len(features_train), len(features_test))
print("Пропуски:\n", frame.isna().sum().loc[lambda count: count > 0])
frame.head()

## 3. Частоты всех символов odor

Редкими считаем символы с долей **строго меньше 10%** от числа строк train.
Список кодов заранее не задан. Таблица ниже описывает полный train;
внутри GridSearchCV частоты будут заново вычисляться на обучающей части
каждого фолда. Test не участвует в расчёте частот.


In [ ]:
from src.data.prepare import describe_odor

odor_frequencies = describe_odor(features_train)
odor_frequencies

## 4. Собственный трансформер и две ветви обработки

RareOdorCounter наследует TransformerMixin и BaseEstimator. Он добавляет
odor_rare_count: поскольку odor — одна буква, счётчик равен 0 либо 1.
Неизвестная буква считается редкой; пропуск сохраняется как NaN.

Числовая ветвь: SimpleImputer(median) → StandardScaler.
Категориальная: SimpleImputer(most_frequent) → OneHotEncoder(ignore).
Модель: LogisticRegression. Следующая ячейка только собирает конвейер.


In [ ]:
from src.models.pipeline import build_pipeline

pipeline = build_pipeline()
pipeline

## 5. Обучение — запускается следующей ячейкой

Вызывается тот же код, что и командой python -m src.models.train.
GridSearchCV выбирает C из [0.1, 1.0, 10.0] по ROC-AUC на пяти фолдах train,
затем лучший конвейер оценивается на отложенном test.

Функция сохраняет модель, разбиения, метрики, прогнозы, частоты и графики;
проверяет совпадение прогнозов после загрузки joblib. Повторный вызов
переобучает модель и обновляет эти файлы.


In [ ]:
from src.models.train import main as train_model

train_model()

## 6. Просмотр результатов собственного запуска

Положительный класс — p (ядовитый). Accuracy измеряет долю верных ответов;
ROC-AUC — качество ранжирования по вероятности p. Сравните средний CV ROC-AUC
и test ROC-AUC; отдельно рассмотрите ошибки на матрице.


In [ ]:
import json

reports_dir = project_root / "reports"
metrics = json.loads((reports_dir / "metrics.json").read_text())
metadata = json.loads((reports_dir / "run_metadata.json").read_text())
print("Лучшие параметры:", metadata["best_params"])
print("Средний CV ROC-AUC:", metadata["best_cv_roc_auc"])
print("Редкие символы:", metadata["rare_symbols"])
metrics

In [ ]:
import pandas as pd

pd.read_csv(reports_dir / "cv_results.csv")

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(reports_dir / "figures" / "evaluation.png")))

## 7. DVC и отчёт

В терминале из корня проекта выполните после загрузки и обучения:

```bash
dvc add data/raw/mushroom.csv
dvc add data/processed/train.csv data/processed/test.csv models/mushroom.joblib
dvc status
```

Добавьте созданные .dvc-файлы в Git по инструкциям README. Данные остаются
в локальном DVC-кэше; удалённое хранилище автоматически не настраивается.

Заполните reports/report_template.md реальными метриками и выводами.
Сделайте скриншоты структуры проекта, dvc status, результатов и интерфейса.
Готовые метрики и скриншоты намеренно не включены в исходный ноутбук.
